# Part B - Low-Rank Adaptation (LoRA)
## Overview
### `Name:`
### `Roll Number:`

References:
https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch


## Instructions

<table style="width:100%; table-layout:fixed;">
<tr>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    Plagiarism Policy
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      <strong>All work must be done independently.</strong><br>
      Any plagiarism or cheating (from peers or the internet) will be immediately referred to the DC.
      If you are unsure what constitutes plagiarism, consult the TAs in a timely manner.
    </li>
    <li style="margin-top:8px;">
      <strong>Do not look at anyone else’s code.</strong>
    </li>
  </ul>
</td>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    Submission Instructions
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      Submit <strong>all</strong> required <code>.ipynb</code> and <code>.py</code> files on LMS
      (search how to extract scripts if confused).
    </li>
    <li style="margin-top:8px;">
      <strong>Submissions via Dropbox or email will not be accepted.</strong>
    </li>
    <li style="margin-top:8px;">
      Zip your files as <code>RollNumber_PAx.zip</code> and ensure your roll number is filled in correctly.
    </li>
    <li style="margin-top:8px;">
      <strong>Deviation from the naming convention will result in a penalty.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>Expected file structure:</strong>
    </li>
  </ul>
  <pre style="margin:8px 0 0 0; font-size:90%; color:#e53935;">
26100076_PA3.zip
├─ 26100076_PA3.ipynb
└─ 26100076_PA3.py
  </pre>
</td>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    General Instructions
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      <strong>Ensure all cells are executed before submission.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>Do not remove or modify any pre-written code.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>All parts of the assignment must be attempted.</strong>
    </li>
  </ul>
</td>

</tr>
</table>

# Low-Rank Adaptation (LoRA) — An Intuitive Explanation

Large language models (LLMs) and vision transformers contain **millions or billions of parameters** (weights).  
Fine-tuning these models normally requires updating *all* of these weights, which is computationally expensive and often limited by GPU memory.

---

### Regular Fine-Tuning

In standard training or fine-tuning:

- Each layer has a large weight matrix **W**
- Training learns a full update matrix **ΔW**
- The updated weights are:

$$
W_{\text{updated}} = W + \Delta W
$$

**Drawback:**  
The update matrix **ΔW** is very large, making training slow and memory-intensive.

---

### LoRA: The Key Idea

LoRA (Low-Rank Adaptation) is based on the observation that:

> Most useful weight updates can be represented by **simple patterns**, rather than a full, complex matrix.

Instead of learning the full **ΔW**, LoRA **approximates it** using two much smaller matrices:

$$
\Delta W \approx A \cdot B
$$

where:
- **A** and **B** are small matrices
- Their matrix product has the same shape as **W**

The weight update becomes:

$$
W_{\text{updated}} = W + A \cdot B
$$

The figure below illustrates these formulas for full finetuning and LoRA side by side.

![LoRA illustration](LoRAImage.webp)

---

### Intuitive Explanation

Think of the model as a large machine with many adjustment knobs.

- **Full fine-tuning:**  
  Adjust every knob individually.

- **LoRA:**  
  Keep the original knobs fixed and add a small attachment that adjusts many knobs together in a coordinated way.

This attachment (matrices **A** and **B**) captures the most important changes while using far fewer parameters.

## Implementing a LoRA Layer
Implement a LoRA layer that adapts a neural network without directly modifying its original weights. Instead of learning a full weight update, represent the adaptation as a low-rank decomposition that captures task-specific changes using a compact set of parameters. This formulation reduces both memory usage and computational cost compared to full fine-tuning.

Design the LoRA layer using two trainable low-rank matrices whose product defines the weight update. Scale this update appropriately and optionally apply dropout to regulate its effect. During the forward pass, compute the low-rank update independently so it can later be combined with the output of a frozen base layer in a modular and efficient manner.

In [92]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class LoRALayer(nn.Module):
    """
    Implements the LoRA low-rank adaptation: ΔW = alpha * B @ A
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha

        # A: (rank, in_features)
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.01)

        # B: (out_features, rank)
        self.B = nn.Parameter(torch.zeros(out_features, rank))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Computes: alpha * x @ A.T @ B.T
        """
        x = self.dropout(x)
        return self.alpha * (x @ self.A.T @ self.B.T)


## LoRA-Adapted Linear Layer (non-merged)
Implement a LoRA-adapted version of a linear layer by wrapping an existing nn.Linear module. The original weights and bias should be frozen to ensure that only the low-rank adaptation is trained. The forward pass should combine the output of the frozen linear layer with the output of the LoRA layer, illustrating how task-specific updates can be added without modifying the base parameters.

In [93]:
class LoRALinear(nn.Module):
    """
    A linear layer with LoRA adaptation.
    """

    def __init__(self, linear_layer, rank=8, alpha=16, dropout=0.0):
        super().__init__()

        self.linear = linear_layer

        # Freeze original weights
        self.linear.weight.requires_grad_(False)
        if self.linear.bias is not None:
            self.linear.bias.requires_grad_(False)

        self.lora = LoRALayer(
            in_features=linear_layer.in_features,
            out_features=linear_layer.out_features,
            rank=rank,
            alpha=alpha,
            dropout=dropout,
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)


## Define a Small Network with One Linear Layer
Create a simple experimental setup using a single linear layer and a randomly generated input tensor. Fix the random seed to ensure reproducibility. Compute and record the output of the original linear layer, which will serve as a baseline for comparison when LoRA is introduced.

In [94]:
# Hyperparameters
random_seed = 123
torch.manual_seed(random_seed)

layer = nn.Linear(10, 2)
x = torch.randn((1, 10))

print("Input x:\n", x)
print("\nOriginal Linear Layer:\n", layer)
print("\nOriginal output:")
print(layer(x))


Input x:
 tensor([[ 0.5490,  0.3671,  0.1219,  0.6466, -1.4168,  0.8429, -0.6307,  1.2340,
          0.3127,  0.6972]])

Original Linear Layer:
 Linear(in_features=10, out_features=2, bias=True)

Original output:
tensor([[0.6639, 0.4487]], grad_fn=<AddmmBackward0>)


## Apply LoRA to the Linear Layer
Apply the LoRA-adapted linear wrapper to the previously defined layer. Run the same input through this modified layer and observe the change in output compared to the baseline. This step demonstrates how LoRA introduces an additional learned transformation while keeping the original layer weights unchanged.

In [95]:
layer_lora = LoRALinear(layer, rank=2, alpha=4)
print("\nOutput with LoRA (unmerged):")
print(layer_lora(x))



Output with LoRA (unmerged):
tensor([[0.6639, 0.4487]], grad_fn=<AddBackward0>)


## Merged LoRA Version (for inference)
Implement an alternative version of the linear layer in which the LoRA weight update is merged directly into the original weight matrix. In this formulation, compute the low-rank weight update and add it to the frozen base weights before applying the linear operation. This merged approach reflects how LoRA can be deployed efficiently during inference.

In [96]:
class LinearWithLoRAMerged(nn.Module):
    """
    Linear layer where LoRA weights are merged into W.
    """

    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear

        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # ΔW = alpha * B @ A
        delta_w = self.lora.alpha * (self.lora.B @ self.lora.A)

        # W' = W + ΔW
        combined_weight = self.linear.weight + delta_w

        return F.linear(x, combined_weight, self.linear.bias)


## Verify Mathematical Equivalence
Verify that the merged and non-merged LoRA implementations produce identical outputs when initialized with the same parameters. Ensure both models share the same low-rank matrices, then compare their outputs on the same input. Report the maximum absolute difference to confirm numerical equivalence between the two approaches.

In [97]:
layer_lora_unmerged = LoRALinear(layer, rank=2, alpha=4)
layer_lora_merged = LinearWithLoRAMerged(layer, rank=2, alpha=4)

# Copy LoRA parameters so both models are identical
layer_lora_merged.lora.A.data = layer_lora_unmerged.lora.A.data.clone()
layer_lora_merged.lora.B.data = layer_lora_unmerged.lora.B.data.clone()

out_unmerged = layer_lora_unmerged(x)
out_merged = layer_lora_merged(x)

print("\nUnmerged LoRA output:")
print(out_unmerged)

print("\nMerged LoRA output:")
print(out_merged)

print("\nMax difference:")
print((out_unmerged - out_merged).abs().max())



Unmerged LoRA output:
tensor([[0.6639, 0.4487]], grad_fn=<AddBackward0>)

Merged LoRA output:
tensor([[0.6639, 0.4487]], grad_fn=<AddmmBackward0>)

Max difference:
tensor(0., grad_fn=<MaxBackward1>)


# Applying LoRA Layers to LLM

In [98]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset


## Preparing the Dataset
Prepare a text dataset suitable for causal language modeling. Load a subset of the AG News dataset, shuffle it for reproducibility, and split it into training and evaluation sets. Tokenize the text with fixed-length padding and truncation, and construct labels that mask padding tokens so they do not contribute to the training loss. Finally, format the dataset for PyTorch and create data loaders for batch-based training and evaluation.

In [99]:
def prepare_dataset(tokenizer, max_samples=500, max_length=128):
    dataset = load_dataset("ag_news")

    train_data = dataset["train"].shuffle(seed=42).select(range(max_samples))
    test_data = dataset["test"].shuffle(seed=42).select(range(int(max_samples * 0.2)))

    def tokenize(example):
        tokenized = tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )

        labels = tokenized["input_ids"].copy()
        labels = [
            -100 if token == tokenizer.pad_token_id else token
            for token in labels
        ]

        tokenized["labels"] = labels
        return tokenized

    train_data = train_data.map(tokenize, remove_columns=train_data.column_names)
    test_data = test_data.map(tokenize, remove_columns=test_data.column_names)

    train_data.set_format("torch")
    test_data.set_format("torch")

    train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=8)

    return train_loader, test_loader


## Initializing the Base Language Model
Load a pre-trained causal language model and its corresponding tokenizer. Configure the tokenizer to use a valid padding token and move the model to the appropriate computation device. Keep this base model in evaluation mode to serve as a frozen reference for comparison with the LoRA-adapted version.

In [100]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
model.eval()


tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

## Creating a LoRA-Adapted Model
Instantiate a second copy of the pre-trained language model that will be modified using LoRA. This model will be used for training and should be set to training mode. Maintaining separate base and LoRA-adapted models allows direct comparison of parameter counts and downstream performance.

In [101]:
model_lora = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
model_lora.train()


GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

## Applying LoRA to Attention Layers
Modify the language model by replacing selected linear layers within the attention mechanism with LoRA-adapted linear layers. Target projection layers commonly used in self-attention, such as query, key, value, and output projections. This step injects low-rank adaptations into the model while leaving the original weights structurally intact.

In [102]:
def apply_lora_to_model(model, rank=8, alpha=16):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and any(
            key in name for key in ["q_lin", "k_lin", "v_lin", "out_lin", "c_attn", "c_proj"]
        ):
            parent = model
            path = name.split(".")
            for p in path[:-1]:
                parent = getattr(parent, p)

            setattr(
                parent,
                path[-1],
                LoRALinear(module, rank=rank, alpha=alpha),
            )


In [104]:
apply_lora_to_model(model_lora, rank=8, alpha=16)


## Freezing Base Parameters and Enabling LoRA Training
Freeze all original model parameters to prevent them from being updated during training. Enable gradient updates only for the LoRA low-rank matrices. This ensures that learning is restricted to the parameter-efficient LoRA components while the pre-trained knowledge of the base model is preserved.

In [105]:
for name, param in model_lora.named_parameters():
    if name.endswith(".A") or name.endswith(".B"):
        param.requires_grad = True
    else:
        param.requires_grad = False


## Inspecting Trainable Parameters
Verify that only the intended LoRA parameters are trainable by inspecting the model’s parameter list. This step helps confirm that the parameter-freezing strategy has been applied correctly before training begins.

In [106]:
def check_trainable_params(model):
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(name)

check_trainable_params(model_lora)


transformer.h.0.mlp.c_proj.lora.A
transformer.h.0.mlp.c_proj.lora.B
transformer.h.1.mlp.c_proj.lora.A
transformer.h.1.mlp.c_proj.lora.B
transformer.h.2.mlp.c_proj.lora.A
transformer.h.2.mlp.c_proj.lora.B
transformer.h.3.mlp.c_proj.lora.A
transformer.h.3.mlp.c_proj.lora.B
transformer.h.4.mlp.c_proj.lora.A
transformer.h.4.mlp.c_proj.lora.B
transformer.h.5.mlp.c_proj.lora.A
transformer.h.5.mlp.c_proj.lora.B
transformer.h.6.mlp.c_proj.lora.A
transformer.h.6.mlp.c_proj.lora.B
transformer.h.7.mlp.c_proj.lora.A
transformer.h.7.mlp.c_proj.lora.B
transformer.h.8.mlp.c_proj.lora.A
transformer.h.8.mlp.c_proj.lora.B
transformer.h.9.mlp.c_proj.lora.A
transformer.h.9.mlp.c_proj.lora.B
transformer.h.10.mlp.c_proj.lora.A
transformer.h.10.mlp.c_proj.lora.B
transformer.h.11.mlp.c_proj.lora.A
transformer.h.11.mlp.c_proj.lora.B


## Comparing Parameter Counts
Compute and compare the total number of parameters and the number of trainable parameters for both the base model and the LoRA-adapted model. Report the percentage reduction in trainable parameters achieved through LoRA to quantify its efficiency benefits.

In [107]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, 100 * trainable / total

base_total, base_trainable, _ = count_parameters(model)
lora_total, lora_trainable, lora_pct = count_parameters(model_lora)

print(f"Base model trainable parameters: {base_trainable:,}")
print(f"LoRA trainable parameters: {lora_trainable:,}")
print(f"Trainable parameter reduction: {100 - lora_pct:.2f}%")


Base model trainable parameters: 125,198,592
LoRA trainable parameters: 368,640
Trainable parameter reduction: 99.71%


## Training the LoRA-Adapted Model
Train the LoRA-adapted model using the prepared dataset and an appropriate optimizer. Perform multiple training epochs while monitoring the training loss. Since only a small subset of parameters is being updated, this training process is significantly more efficient than full fine-tuning.

In [108]:
train_loader, test_loader = prepare_dataset(tokenizer)

optimizer = torch.optim.AdamW(
    [p for p in model_lora.parameters() if p.requires_grad],
    lr=2e-4
)

model_lora.train()

for epoch in range(2):
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model_lora(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            print(f"Epoch {epoch} | Step {step} | Loss {loss.item():.4f}")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Epoch 0 | Step 0 | Loss 4.0015
Epoch 0 | Step 50 | Loss 3.9422
Epoch 1 | Step 0 | Loss 3.9204
Epoch 1 | Step 50 | Loss 3.7683


## Evaluating Model Accuracy
Evaluate both the original base model and the LoRA-adapted model on the test dataset. Compute token-level accuracy while ignoring masked positions. Compare the results to assess how well the LoRA-based fine-tuning performs relative to the frozen pre-trained model.

In [109]:
@torch.no_grad()
def compute_accuracy(model, dataloader, device):
    model.eval()
    correct, total = 0, 0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits

        preds = logits.argmax(dim=-1)
        labels = batch["labels"]

        mask = labels != -100
        correct += ((preds == labels) & mask).sum().item()
        total += mask.sum().item()

    return 100 * correct / total


In [ ]:
print(f"Test accuracy orig model: {compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

Test accuracy orig model: 0.21%
Test accuracy LoRA model: 0.36%


# Reflective Questions

Q1. Why is representing weight updates as a low-rank decomposition effective for large language models? \
Q2. What assumptions does LoRA make about the nature of task-specific adaptations?\
Q3. Why are LoRA layers typically applied to attention projection layers rather than all linear layers?\
Q4. How does the choice of rank affect the expressive capacity of the LoRA adaptation?\
Q5. Why might a LoRA-adapted model perform comparably to a fully fine-tuned model on some tasks?\
Q6. Under what conditions could LoRA lead to worse performance than full fine-tuning?

References:
https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch